In [ ]:
import glob
import os
import sys
import numpy as np
# os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
	# Mount Google Drive if running in Colab
	ROOT_DATA = '/content/dataset/pump/'
	sys.path.append(os.path.abspath('.'))
else:
	ROOT_DATA = '../data/raw/pump/'
	sys.path.append(os.path.abspath('..'))

search_pattern = os.path.join(ROOT_DATA, 'id_*/normal/*.wav')
all_files = glob.glob(search_pattern)
print(f'Total files found: {len(all_files)}')

from src.utils import AudioPreprocessor

preprocessor = AudioPreprocessor()

Total files found: 3749


In [4]:
import librosa

def data_generator(files, batch_size=32):
	while True:
		np.random.shuffle(files)
		for i in range(0, len(files), batch_size):
			batch_files = files[i:i+batch_size]
			batch_data = []

			for file in batch_files:
				mel_spec = preprocessor.transform(file)

				if mel_spec.shape[1] != 313:
					mel_spec = librosa.util.fix_length(mel_spec, size=313, axis=1)
				mel_spec = mel_spec[..., np.newaxis]
				batch_data.append(mel_spec)

			batch_x = np.array(batch_data)
			yield batch_x, batch_x
			del batch_data

train_dataset = tf.data.Dataset.from_generator(
	lambda: data_generator(all_files, batch_size=32),
	output_signature=(
		tf.TensorSpec(shape=(None, 128, 313, 1), dtype=tf.float32),
		tf.TensorSpec(shape=(None, 128, 313, 1), dtype=tf.float32)
	)
).prefetch(tf.data.AUTOTUNE)

In [5]:
from src.model import build_autoencoder

model = build_autoencoder()

checkpoint = ModelCheckpoint(
	'best_model.keras',
	monitor='loss',
	save_best_only=True,
	verbose=1
)

history = model.fit(
	train_dataset,
	epochs=20,
	steps_per_epoch=len(all_files) // 32,
	callbacks=[checkpoint]
)

print("\nTraining Complete. Model saved as 'best_model.keras'")

Epoch 1/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0366
Epoch 1: loss improved from inf to 0.01833, saving model to best_model.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 167s 1s/step - loss: 0.0365
Epoch 2/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0055
Epoch 2: loss improved from 0.01833 to 0.00533, saving model to best_model.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 139s 1s/step - loss: 0.0055
Epoch 3/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0048
Epoch 3: loss improved from 0.00533 to 0.00469, saving model to best_model.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - loss: 0.0048
Epoch 4/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0043
Epoch 4: loss improved from 0.00469 to 0.00422, saving model to best_model.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 129s 1s/step - loss: 0.0043
Epoch 5/20
117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - loss: 0.0041
Epoch 5: loss improved from 0.00422 to 0.00398, saving model to best_model.keras
117/117 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step